# Chạy ứng dụng CreditScore (Streamlit) trên Google Colab

Ứng dụng hỗ trợ cán bộ tín dụng ở giai đoạn 2 của đồ án, gồm 3 trang: **Quản lý mô hình**, **So sánh mô hình** và **Chấm điểm hồ sơ**. Trang Chấm điểm hồ sơ: nhập 12 biến của hồ sơ (cho phép thiếu), tính **PD đã hiệu chỉnh**,
đưa ra **khuyến nghị minh họa** (Phê duyệt / Xem xét thủ công / Từ chối) và giải thích bằng **SHAP** theo 12 biến gốc.
Ứng dụng dùng đúng mô hình đã lưu trong repository (`models/champion.joblib`).

### Các bước
1. **Runtime → Run all.** Lần đầu, ô 1 cài thư viện rồi tự khởi động lại phiên; chọn lại **Runtime → Run all**.
2. Ô 2 tìm repository (sửa `REPO_URL` nếu repository công khai; nếu riêng tư, tải lên file ZIP tải từ GitHub).
3. Ô 3 khởi động ứng dụng và in ra **đường dẫn**; bấm vào để mở ứng dụng trong tab mới (hoặc xem ngay trong khung bên dưới ô 4).

### Giới hạn
- Đường dẫn chỉ dùng được **khi phiên Colab còn chạy** và chỉ mở được bằng **tài khoản Google của bạn**; không gửi cho người khác được.
- Phiên Colab miễn phí tự ngắt khi không tương tác (khoảng 90 phút) hoặc tối đa khoảng 12 giờ; khi đó chạy lại notebook.
- Muốn chia sẻ công khai, xem hướng dẫn triển khai Streamlit Community Cloud trong `README.md`.

Các nhãn khuyến nghị là **minh họa học thuật** với ngưỡng 10% và 50% (điều chỉnh được), không phải quyết định cấp tín dụng.

## 1. Cài đúng phiên bản thư viện
Mô hình đã lưu cần đúng phiên bản scikit-learn và LightGBM. Ô này chỉ cài khi thiếu, rồi tự khởi động lại phiên.

In [ ]:
import importlib.metadata as md, os, subprocess, sys
PINS = {"numpy": "2.4.4", "pandas": "3.0.2", "scipy": "1.17.1", "scikit-learn": "1.8.0", "lightgbm": "4.7.0",
        "xgboost": "3.2.0", "imbalanced-learn": "0.14.2", "scikit-optimize": "0.10.2", "shap": "0.51.0",
        "streamlit": "1.64.0", "altair": "6.3.0"}
def ok(p, v):
    try:
        return md.version(p) == v
    except md.PackageNotFoundError:
        return False
need = [f"{p}=={v}" for p, v in PINS.items() if not ok(p, v)]
if need:
    print("Đang cài:", " ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Cài xong — phiên sẽ khởi động lại. Hãy chọn lại Runtime → Run all.")
    os.kill(os.getpid(), 9)
print("Thư viện đã đúng phiên bản.")

## 2. Tìm repository

In [ ]:
import zipfile
from pathlib import Path
REPO_URL = "https://github.com/TEN-TAI-KHOAN/Apply-ML-algorithms-in-credit-scoring-modelling.git"   # sửa TEN-TAI-KHOAN (repository công khai)

def find_root(base):
    for p in [base, *base.parents, *sorted(base.glob("*")), *sorted(base.glob("*/*"))]:
        if (p / "app" / "streamlit_app.py").exists() and (p / "models" / "champion.joblib").exists():
            return p
    return None

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
ROOT = find_root(Path.cwd()) or find_root(BASE)
if ROOT is None and "TEN-TAI-KHOAN" not in REPO_URL:
    r = subprocess.run(["git", "clone", "-q", REPO_URL, str(BASE / "Apply-ML-algorithms-in-credit-scoring-modelling")], capture_output=True, text=True)
    if r.returncode != 0:             # thường gặp: repository đang Private, hoặc sai tên tài khoản / tên repository
        print("Không clone được repository:", r.stderr.strip().splitlines()[-1] if r.stderr.strip() else r.returncode)
        print("→ Chuyển sang cách tải file ZIP (GitHub → Code → Download ZIP).")
    ROOT = find_root(BASE)
if ROOT is None:                      # repository riêng tư: GitHub → Code → Download ZIP, rồi tải file ZIP lên
    from google.colab import files
    zp = next(iter(files.upload()))
    zipfile.ZipFile(zp).extractall(BASE)
    ROOT = find_root(BASE)
assert ROOT is not None, "Không tìm thấy app/streamlit_app.py và models/champion.joblib"
os.chdir(ROOT)
print("Repository:", ROOT)

## 3. Khởi động ứng dụng
Ứng dụng chạy nền ở cổng 8501. Ô này chờ ứng dụng sẵn sàng rồi in đường dẫn.

In [ ]:
import time, urllib.request
log = open("/tmp/streamlit.log", "w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py", "--server.port", "8501",
                         "--server.headless", "true", "--browser.gatherUsageStats", "false"], stdout=log, stderr=log)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=1); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ứng dụng chưa khởi động được — xem /tmp/streamlit.log:\n" + open("/tmp/streamlit.log").read()[-2000:])
try:
    from google.colab.output import eval_js
    print("Mở ứng dụng tại:", eval_js("google.colab.kernel.proxyPort(8501)"))
except ImportError:
    print("Mở ứng dụng tại: http://localhost:8501")

## 4. (Tùy chọn) Xem ứng dụng ngay trong notebook

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8501, height=900)
except ImportError:
    print("Chỉ dùng được trên Colab.")

## 5. Dừng ứng dụng
Chạy ô này khi dùng xong (hoặc chỉ cần đóng phiên Colab).

In [ ]:
proc.terminate(); print("Đã dừng ứng dụng.")